# 2. Metabolic modelling of microbial communities

## Authors
* Sara Benito Vaquerizo, Genome Biology Unit, European Molecular Biology Laboratory (EMBL), Heidelberg

## Learning outcomes

In this tutorial you will use SteadierCom (https://github.com/cdanielmachado/SteadierCom); a python package to model microbial communities

- **2.1**: Simulate growth of the algae-bacterial community and inspect metabolite dynamics
- **2.2**: Simulate growth of the community under specified relative species abundances
- **2.3**: Simulate growth of the community under a specified medium

## Setup

In [1]:
# Import packages
from steadiercom.steadiercom import SteadierCom
from steadiercom.cli import main_run as run_steadiercom
from reframed.community.model import Community
from reframed.io.cache import ModelCache
from reframed import Environment
from reframed.solvers.solution import Status
import os
import pandas as pd

## 2.1 Simulate growth of the algae-bacterial community and inspect metabolite dynamics
To simulate growth of a community, we are going to use SteadierCom. SteadierCom computes the relative abundance of species for a fixed growth rate of the community, or computes the maximum growth rate for a given set of relative abundaces. 

By default, steadiercom computes the fluxes in the community for a fixed growth rate of 0.1 h-1 and a default complete medium

First, define the models of the species in the community 

In [2]:
species1='../Data/Chaetoceros_gracilis.xml' #Algae
species2='../Data/Limnobacter_thiooxidans.xml' 
species3='../Data/Halopseudomonas.xml'
species4='../Data/Yoonia_vestfoldensis.xml'

#Run steadiercom to assess if the community is feasible for the default medium and growth rate
growth = 0.1
df = run_steadiercom(
    models=[species1, species2, species3, species4], 
    growth=growth, 
    w_e=0.002, 
    w_r=0.2, 
    output=f"../Output/Extracellular_fluxes_gr{growth:.2f}_4species"
)

simulating all in complete medium


/home/bartosz/Documents/Heidelberg/EMBL/Bioinformatician/Courses/MCD2026/Practical_Metabolic_modelling/.pixi/envs/default/lib/python3.14/site-packages/reframed/core/elements.py:62: UserWarning: Atomic weight not listed for elements: {'Z'}
  warn(f"Atomic weight not listed for elements: {missing}")


We can increase the growth rate and see what would be the maximum growth rate where the community would still be feasible

In [3]:
#Increase the growth rate up to 0.3 ,0.5, 1 h-1 and inspect the Output file in ../Output/fluxes_grx_4species.tsv
growth = 1
df = run_steadiercom(
    models=[species1, species2, species3, species4], 
    growth=1, 
    w_e=0.002, 
    w_r=0.2, 
    output=f"../Output/Extracellular_fluxes_gr_{growth:.2f}_4species"
)

simulating all in complete medium


/home/bartosz/Documents/Heidelberg/EMBL/Bioinformatician/Courses/MCD2026/Practical_Metabolic_modelling/.pixi/envs/default/lib/python3.14/site-packages/reframed/core/elements.py:62: UserWarning: Atomic weight not listed for elements: {'Z'}
  warn(f"Atomic weight not listed for elements: {missing}")


Check the output dataframe (on the screen or in `../Output/fluxes_gr_growth_4species.tsv`. What are the cross-feeding metabolites between species? Do you observe any changes in the species present? Would you say the community is stable or are there species that take over?


Modify the growth rates as before, what are the most abundant species in the community?

In [4]:
#We are going to compute the relative abundance for a fixed community growth rate.

files = [species1, species2, species3, species4]
ids = [os.path.basename(f)[:-4] for f in files]   # strip .xml, same as steadiercom.cli does

model_cache = ModelCache(ids, files, load_args={'flavor': 'bigg'})
models = [model_cache.get_model(i, reset_id=True) for i in ids]
community = Community('comm', models, copy_models=False)

env = Environment.complete(community.merged_model, inplace=False)  # complete default medium, as before

sol = SteadierCom(
    community, 
    growth=0.3, 
    allocation=True,
    constraints=env, 
    w_e=0.002, 
    w_r=0.2
)

sol.growth        #Community growth rate
sol.abundance      #Species relative abundance

/home/bartosz/Documents/Heidelberg/EMBL/Bioinformatician/Courses/MCD2026/Practical_Metabolic_modelling/.pixi/envs/default/lib/python3.14/site-packages/reframed/core/elements.py:62: UserWarning: Atomic weight not listed for elements: {'Z'}
  warn(f"Atomic weight not listed for elements: {missing}")


{'Chaetoceros_gracilis': 0.1267775566018069,
 'Limnobacter_thiooxidans': 0.3617056496966632,
 'Halopseudomonas': 0.04771984884919612,
 'Yoonia_vestfoldensis': 0.4637969448523338}

We can store the solution in corresponding dataframes and save it as files to visualize the outputs

In [5]:
abundance_df = (
    pd.DataFrame.from_dict(sol.abundance, orient="index", columns=["abundance"])
    .reset_index(names="organism")
)

growth=0.3

flux_rows = [
    {'organism': org_id, 'reaction': r_id, 'flux': flux}
    for org_id, fluxes in sol.internal.items()
    for r_id, flux in fluxes.items()
]
fluxes_df = pd.DataFrame(flux_rows)

# optional: save alongside your other outputs
abundance_df.to_csv(f'../Output/species_abundances_gr_{growth:.2f}.tsv', sep='\t', index=False)
fluxes_df.to_csv(f'../Output/all_fluxes_gr_{growth:.2f}.tsv', sep='\t', index=False)

## 2.2 Simulate growth of the community under specified relative species abundances

Create a file with defined relative abundances per species based on 16SrRNA sequencing

In [6]:
files = [species1, species2, species3, species4]
ids = [os.path.basename(f)[:-4] for f in files]  # matches extract_id_from_filepath (remove .xml)
abundances = [0.3, 0.4, 0.2, 0.1]                # ~Xenic cullture after 7 days  

pd.DataFrame({
    'community': 'comm1',
    'organism': ids,
    'abundance': abundances,
}).to_csv('../Data/communities_withabundances.tsv', sep='\t', header=False, index=False)

Now we have defined the relative abundance we are going to compute the growth of the 
community under those constraints 

In [7]:
df = run_steadiercom(
    models=files,
    communities='../Data/communities_withabundances.tsv',
    growth=None,          # growth will be maximixed for the fixed abundances
    w_e=0.002,
    w_r=0.2,
    output='../Output/Extracellular_fluxes_fixed_abundance_4species',
)
df

simulating comm1 in complete medium


/home/bartosz/Documents/Heidelberg/EMBL/Bioinformatician/Courses/MCD2026/Practical_Metabolic_modelling/.pixi/envs/default/lib/python3.14/site-packages/reframed/core/elements.py:62: UserWarning: Atomic weight not listed for elements: {'Z'}
  warn(f"Atomic weight not listed for elements: {missing}")


,donor,receiver,compound,rate,mass_rate,community,medium
7,Yoonia_vestfoldensis,environment,M_glc__D_e,19.938066,3.591950e+00,comm1,complete
102,environment,Yoonia_vestfoldensis,M_cellb_e,10.000000,3.422956e+00,comm1,complete
123,environment,Limnobacter_thiooxidans,M_gthrd_e,10.000000,3.063149e+00,comm1,complete
122,Limnobacter_thiooxidans,environment,M_gthox_e,5.000000,3.053070e+00,comm1,complete
15,Limnobacter_thiooxidans,Yoonia_vestfoldensis,M_h2o_e,9.614895,1.732143e-01,comm1,complete
...,...,...,...,...,...,...,...
106,environment,Limnobacter_thiooxidans,M_cobalt2_e,0.000006,3.383337e-07,comm1,complete
162,environment,Yoonia_vestfoldensis,M_zn2_e,0.000005,3.200302e-07,comm1,complete
107,environment,Halopseudomonas,M_cobalt2_e,0.000003,1.691669e-07,comm1,complete
108,environment,Yoonia_vestfoldensis,M_cobalt2_e,0.000001,8.458343e-08,comm1,complete


What are the cross-feeding metabolites? 

Compute the maximum growth rate:

In [8]:
ids = [os.path.basename(f)[:-4] for f in files]
abundance = dict(zip(ids, [0.4, 0.3, 0.2, 0.1]))   # your fixed relative abundances

model_cache = ModelCache(ids, files, load_args={'flavor': 'bigg'})
models = [model_cache.get_model(i, reset_id=True) for i in ids]
community = Community('comm1', models, copy_models=False)

env = Environment.complete(community.merged_model, inplace=False)

sol = SteadierCom(community, abundance=abundance, growth=None,
                   allocation=True, constraints=env, w_e=0.002, w_r=0.2)

if sol.status == Status.OPTIMAL:
    print(f'Community growth rate: {sol.growth:.4f} h⁻¹')
else:
    print(f'No feasible solution — status: {sol.status}')

Community growth rate: 0.1079 h⁻¹


## 2.3 Simulate growth of the community under a specified medium

Before, we have predicted the microbial interactions in the community under a complete rich medium. Now we are going to work
with a user-defined medium to simulate photoautotrophic growth in the algae-bacteria community (medium phaeo in `../Data/media_db.tsv`)

In [9]:
df = run_steadiercom(
    models=files, 
    communities='../Data/communities_withabundances.tsv',      # file with relative abundance
    #communities='../Data/abundance_algae.tsv',
    output='../Output/Extracellular_fluxes_fixed_abundance_marine_medium', 
    media='marine',            # Name of the photoautotrophic medium
    mediadb='../Data/mylibrary.tsv',          # path to media
    #growth=0.1,           # We are going to compute the growth
    w_e=0.002,             # enzyme-sector weight (--we default)
    w_r=0.2,               # ribosome-sector weight (--wr default)
)

df  # pandas DataFrame of cross-feeding results (also written to <output>.tsv)

simulating comm1 in marine medium
No feasible solutions found.
